# Yoruba (New Testament)

In [1]:
"""
Optimized synthesis script for EveryVoice.
Loads models once and synthesizes all sentences in batches.
"""

import os
from pathlib import Path

import torch
import pandas as pd


# ---- Import EveryVoice internals ----
from everyvoice.model.feature_prediction.FastSpeech2_lightning.fs2.model import FastSpeech2
from everyvoice.model.feature_prediction.FastSpeech2_lightning.fs2.cli.synthesize import (
    synthesize_helper,
    get_global_step,
)
from everyvoice.model.feature_prediction.FastSpeech2_lightning.fs2.type_definitions import (
    SynthesizeOutputFormats,
)
from everyvoice.config.type_definitions import DatasetTextRepresentation
from everyvoice.model.vocoder.HiFiGAN_iSTFT_lightning.hfgl.utils import (
    load_hifigan_from_checkpoint,
)
from everyvoice.utils.heavy import get_device_from_accelerator

# ---- Load models ONCE ----
device = get_device_from_accelerator("gpu")
print(f"Using device: {device}")

/home/mila/g/guzmand/scratch/.conda/envs/EveryVoice/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/mila/g/guzmand/scratch/.conda/envs/EveryVoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda:0


In [ ]:
# ---- Configuration ----
TEST_PATH = '/home/mila/g/guzmand/scratch/datasets/bible-tts-resources/Yoruba/validation.tsv'
feature_prediction_checkpoint = Path("logs_and_checkpoints/FeaturePredictionExperiment/base/checkpoints/last.ckpt")
vocoder_base_checkpoint = Path("/home/mila/g/guzmand/scratch/checkpoints/hifigan_universal_v1_everyvoice.ckpt")
output_dir = Path("synthesis_output/yoruba-nt")
wav_dir = output_dir / "wav"

# ---- Read test data ----
test = pd.read_csv(TEST_PATH, sep='\t')
test = test.rename(columns={'filename': 'file'})
test = test[["file", "text"]]
print(f"Total sentences to synthesize: {len(test)}")

In [ ]:

print("Loading feature prediction model...")
model = FastSpeech2.load_from_checkpoint(str(feature_prediction_checkpoint)).to(device)
model.eval()
global_step = get_global_step(feature_prediction_checkpoint)

print("Loading vocoder...")
vocoder_ckpt = torch.load(str(vocoder_base_checkpoint), map_location=device, weights_only=True)
vocoder_model, vocoder_config = load_hifigan_from_checkpoint(vocoder_ckpt, device)
vocoder_global_step = get_global_step(vocoder_base_checkpoint)

print("Models loaded successfully!")

# ---- Prepare data with custom basenames ----
DEFAULT_LANGUAGE = next(iter(model.lang2id.keys()), None)
DEFAULT_SPEAKER = next(iter(model.speaker2id.keys()), None)

filelist_data = []
for _, row in test.iterrows():
    basename = os.path.splitext(row["file"])[0]
    filelist_data.append({
        "basename": basename,
        "characters": row["text"],
        "language": DEFAULT_LANGUAGE,
        "speaker": DEFAULT_SPEAKER,
        "duration_control": 1.0,
    })

print(f"Prepared {len(filelist_data)} entries for synthesis")

# ---- Run batch synthesis ----
print("\nStarting batch synthesis...")

synthesize_helper(
    model=model,
    texts=None,
    style_reference=None,
    language=None,
    speaker=None,
    duration_control=1.0,
    global_step=global_step,
    output_type=[SynthesizeOutputFormats.wav],
    text_representation=DatasetTextRepresentation.characters,
    accelerator="gpu",
    devices="auto",
    device=device,
    batch_size=4,
    num_workers=4,
    filelist=None,
    filelist_data=filelist_data,
    output_dir=output_dir,
    teacher_forcing_directory=None,
    vocoder_model=vocoder_model,
    vocoder_config=vocoder_config,
    vocoder_global_step=vocoder_global_step,
)

print("Batch synthesis complete!")

# ---- Rename output files ----
# EveryVoice names outputs as: basename--speaker--language--ckpt=N--v_ckpt=M--pred.wav
# We rename them to just: basename.wav
wav_files = list(wav_dir.glob("*.wav")) if wav_dir.exists() else []
print(f"Found {len(wav_files)} generated wav files")

renamed = 0
for wav_path in wav_files:
    parts = wav_path.name.split("--")
    original_basename = parts[0]
    target_name = f"{original_basename}.wav"
    target_path = wav_dir / target_name

    if wav_path.name != target_name:
        os.rename(wav_path, target_path)
        renamed += 1

print(f"Renamed {renamed} files")

# Verify all expected files exist
missing = []
for _, row in test.iterrows():
    expected_path = wav_dir / row["file"]
    if not expected_path.exists():
        missing.append(row["file"])

if missing:
    print(f"\nMissing {len(missing)} files:")
    for f in missing[:10]:
        print(f"  - {f}")
    if len(missing) > 10:
        print(f"  ... and {len(missing) - 10} more")
else:
    print(f"\nAll {len(test)} expected wav files are present!")
